In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

RANDOM_STATE = 42


In [8]:
import glob, os, pandas as pd

RAW_PATH = "../data/raw/train_test_network.csv"
PROCESSED_GLOBS = [
    "../data/processed/**/Network_dataset_*.csv",
    "../data/processed/Network_dataset_*.csv",
]

# ---------- CONFIG (CHỐT) ----------
SAMPLE_FRAC = 0.15  # Sample 15% from each file (reduced for memory efficiency)
# ----------------------------------

files_found = []
for pat in PROCESSED_GLOBS:
    files_found.extend(glob.glob(pat, recursive=True))

files_found = sorted(list(dict.fromkeys(files_found)))  # unique + stable

if len(files_found) > 0:
    print(f"Found {len(files_found)} processed network files.")
    print(f"Strategy: Sampling {SAMPLE_FRAC*100:.0f}% from each of {len(files_found)} files")
    print("Loading and sampling...")

    df_list = []
    for i, f in enumerate(files_found, 1):
        print(f"  [{i}/{len(files_found)}] {os.path.basename(f)}")
        df_part = pd.read_csv(f, low_memory=False)
        df_sampled = df_part.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
        df_list.append(df_sampled)

    df = pd.concat(df_list, ignore_index=True)
    files_to_use = files_found
    data_mode = f"processed_stratified_sample_{len(files_found)}files_frac{SAMPLE_FRAC}"
else:
    print("No processed dataset found → fallback to raw file")
    df = pd.read_csv(RAW_PATH)
    files_to_use = [RAW_PATH]
    data_mode = "raw_single"

print("\nDataset loaded")
print("Mode:", data_mode)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


Found 23 processed network files.
Strategy: Sampling 15% from each of 23 files
Loading and sampling...
  [1/23] Network_dataset_1.csv
  [2/23] Network_dataset_10.csv
  [3/23] Network_dataset_11.csv
  [4/23] Network_dataset_12.csv
  [5/23] Network_dataset_13.csv
  [6/23] Network_dataset_14.csv
  [7/23] Network_dataset_15.csv
  [8/23] Network_dataset_16.csv
  [9/23] Network_dataset_17.csv
  [10/23] Network_dataset_18.csv
  [11/23] Network_dataset_19.csv
  [12/23] Network_dataset_2.csv
  [13/23] Network_dataset_20.csv
  [14/23] Network_dataset_21.csv
  [15/23] Network_dataset_22.csv
  [16/23] Network_dataset_23.csv
  [17/23] Network_dataset_3.csv
  [18/23] Network_dataset_4.csv
  [19/23] Network_dataset_5.csv
  [20/23] Network_dataset_6.csv
  [21/23] Network_dataset_7.csv
  [22/23] Network_dataset_8.csv
  [23/23] Network_dataset_9.csv

Dataset loaded
Mode: processed_stratified_sample_23files_frac0.15
Rows: 3350853
Columns: 47


In [9]:
# Labels + features
assert "label" in df.columns, "ERROR: label column not found"

# Drop leakage-prone / identifier columns if present (Stage 1 policy)
DROP_COLS = ["src_ip", "dst_ip", "type", "ts"]
drop_existing = [c for c in DROP_COLS if c in df.columns]
if drop_existing:
    df = df.drop(columns=drop_existing)
    print("Dropped leakage-prone columns:", drop_existing)

y = df["label"].astype(int)
X = df.drop(columns=["label"])

# Memory optimization: convert numeric columns to more efficient dtypes
for col in X.select_dtypes(include=['float64']).columns:
    X[col] = X[col].astype('float32')
for col in X.select_dtypes(include=['int64']).columns:
    if col != 'label':  # already handled

        X[col] = X[col].astype('int32')

print(f"Memory usage: {X.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("Label distribution (full dataset):")
print("X shape:", X.shape)

display(y.value_counts(normalize=True).mul(100).round(2))

Dropped leakage-prone columns: ['src_ip', 'dst_ip', 'type', 'ts']
Memory usage: 5066.0 MB
Label distribution (full dataset):
X shape: (3350853, 42)


label
1    96.43
0     3.57
Name: proportion, dtype: float64

In [10]:
# Step 1: split train (70%) and temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# Step 2: split temp into val (15%) and test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Split sizes:")
print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")


Split sizes:
Train: 2345597 samples
Validation: 502628 samples
Test: 502628 samples


In [16]:
def label_stats(y, name):
    stats = y.value_counts(normalize=True).mul(100).round(2)
    print(f"{name} label distribution (%):")
    display(stats)

label_stats(y_train, "Train")
label_stats(y_val, "Validation")
label_stats(y_test, "Test")


Train label distribution (%):


label
1    96.43
0     3.57
Name: proportion, dtype: float64

Validation label distribution (%):


label
1    96.43
0     3.57
Name: proportion, dtype: float64

Test label distribution (%):


label
1    96.43
0     3.57
Name: proportion, dtype: float64

In [17]:
imbalance_ratio = y.value_counts()[1] / y.value_counts()[0]

print(f"Attack-to-normal ratio (full dataset): {imbalance_ratio:.2f}")
print("→ This imbalance motivates the use of recall- and PR-based evaluation metrics.")


Attack-to-normal ratio (full dataset): 27.04
→ This imbalance motivates the use of recall- and PR-based evaluation metrics.


In [21]:
# Save split indices (positions) for reproducibility across notebooks
import os, json
SPLIT_DIR = "../outputs/splits/"
os.makedirs(SPLIT_DIR, exist_ok=True)

# Because we used ignore_index=True (or reset_index), index is a stable RangeIndex.
np.save(f"{SPLIT_DIR}/train_idx.npy", X_train.index.values)
np.save(f"{SPLIT_DIR}/val_idx.npy", X_val.index.values)
np.save(f"{SPLIT_DIR}/test_idx.npy", X_test.index.values)

meta = {
    "random_state": RANDOM_STATE,
    "mode": data_mode,
    "n_samples_total": int(len(df)),
    "n_train": int(len(X_train)),
    "n_val": int(len(X_val)),
    "n_test": int(len(X_test)),
    "label_attack_pct_total": float(y.mean()*100),
    "label_attack_pct_train": float(y_train.mean()*100),
    "label_attack_pct_val": float(y_val.mean()*100),
    "label_attack_pct_test": float(y_test.mean()*100),
}

with open(f"{SPLIT_DIR}/split_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Split indices saved to outputs/splits/")
print("Saved metadata to outputs/splits/split_meta.json")


Split indices saved to outputs/splits/
Saved metadata to outputs/splits/split_meta.json


In [22]:
import json, os

SPLIT_DIR = "../outputs/splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

manifest = {
    "data_mode": data_mode,
    "files_used": files_to_use,
    "sample_frac": SAMPLE_FRAC,
    "n_rows": int(len(df)),
    "random_state": RANDOM_STATE,
    "split": "stratified_70_15_15",
}
with open(os.path.join(SPLIT_DIR, "data_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved manifest to:", os.path.join(SPLIT_DIR, "data_manifest.json"))

Saved manifest to: ../outputs/splits\data_manifest.json


In [23]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "samples": [
        X_train.shape[0],
        X_val.shape[0],
        X_test.shape[0]
    ],
    "attack_percentage (%)": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100
    ]
}).round(2)

display(split_summary)


,split,samples,attack_percentage (%)
0,train,2345597,96.43
1,validation,502628,96.43
2,test,502628,96.43
